# Code mẫu — Multimodal

Ghép kết quả từ nhiều nguồn dữ liệu đòi hỏi ba điều: ghép đúng mẫu theo ID, đúng thứ tự các lớp, và có cách xử lý khi một nguồn bị thiếu. Ta bắt đầu bằng một bảng xác suất kiểm được bằng tay. Sau đó ta train hai nhánh trên hai phép đo tổng hợp, rồi chọn cách hiệu chỉnh xác suất và trọng số ghép bằng một tập dữ liệu riêng.

## Ví dụ 1: Ghép theo ID, và căn lại thứ tự các lớp

Ba mẫu, mỗi mẫu gồm một ảnh và một chú thích, cần phân loại là “xe” hay “chuông”. Bảng xác suất của hai nhánh được đặt bằng tay. Nhánh ảnh xếp lớp theo thứ tự `xe, chuông`. Nhánh chữ lại xếp `chuông, xe`, và lưu các ID theo thứ tự khác. Ta ghép muộn, tức lấy trung bình có trọng số của hai bảng xác suất, với trọng số 0.25 cho ảnh và 0.75 cho chữ.

In [ ]:
import numpy as np

ids = ["notice-a", "notice-b", "notice-c"]
image_by_id = {
    "notice-a": [0.8, 0.2],
    "notice-b": [0.3, 0.7],
    "notice-c": [0.6, 0.4],
}
text_by_id = {
    "notice-c": [0.8, 0.2],
    "notice-a": [0.7, 0.3],
    "notice-b": [0.1, 0.9],
}
image_classes = ["xe", "chuong"]
text_classes = ["chuong", "xe"]
assert set(image_by_id) == set(text_by_id) == set(ids)
assert set(image_classes) == set(text_classes)

image_p = np.array([image_by_id[sample_id] for sample_id in ids])
text_p = np.array([text_by_id[sample_id] for sample_id in ids])
column_order = [text_classes.index(label) for label in image_classes]
aligned_text_p = text_p[:, column_order]
fused_p = 0.25 * image_p + 0.75 * aligned_text_p
assert np.allclose(fused_p.sum(axis=1), 1)

print("thu tu lop:", image_classes)
for sample_id, probability in zip(ids, fused_p):
    label = image_classes[probability.argmax()]
    print(sample_id, np.round(probability, 3), "->", label)

thu tu lop: ['xe', 'chuong']
notice-a [0.425 0.575] -> chuong
notice-b [0.75 0.25] -> xe
notice-c [0.3 0.7] -> chuong


Lấy `notice-a` làm ví dụ. Nhánh ảnh cho xe 0.8. Nhánh chữ cho xe 0.3, nằm ở cột thứ hai của nó. Xác suất xe sau khi ghép là $0.25\cdot0.8+0.75\cdot0.3=0.425$. Nếu cộng thẳng cột đầu của hai bảng, ta đã trộn “xe” của nhánh này với “chuông” của nhánh kia, và ra một kết quả vô nghĩa.

Có hai bước sửa, mỗi bước một việc. Ghép theo ID làm cho mỗi hàng đúng là cùng một mẫu. Căn lại thứ tự lớp làm cho mỗi cột đúng là cùng một lớp. Shape của hai bảng khớp nhau không thay được hai bước này.

## Ví dụ 2: Một nguồn kém tin cậy, hoặc bị thiếu

Bảng có bốn mẫu, mỗi hàng là xác suất lớp 1 do hai nhánh đưa ra. Trọng số gốc 0.7 và 0.3 đã chọn từ trước. Ở hàng đầu, một cờ chất lượng hạ độ tin cậy của nhánh thứ nhất. Nhánh bị thiếu được ghi là `NaN`. Ta sẽ thấy vì sao không được nhân `NaN` với 0 rồi cộng như thường.

In [ ]:
import numpy as np

probabilities = np.array([
    [0.1, 0.9],
    [0.8, np.nan],
    [np.nan, 0.3],
    [np.nan, np.nan],
])
base_weights = np.array([0.7, 0.3])
reliability = np.array([
    [0.1, 1.0],
    [1.0, 1.0],
    [1.0, 1.0],
    [1.0, 1.0],
])
available = np.isfinite(probabilities)
weights = base_weights * reliability * available
denominator = weights.sum(axis=1)
usable = denominator > 0
fused = np.full(len(probabilities), np.nan)
safe_values = np.where(available, probabilities, 0.0)
fused[usable] = (safe_values * weights).sum(axis=1)[usable] / denominator[usable]

assert np.allclose(fused[:3], [0.277 / 0.37, 0.8, 0.3])
assert np.isnan(fused[3])
print(f"hang 1, trong so con lai: {weights[0].round(2)}")
for row, value in enumerate(fused, start=1):
    print(f"hang {row}: p1={value:.3f}" if np.isfinite(value)
          else f"hang {row}: REVIEW (khong con nguon)")

hang 1, trong so con lai: [0.07 0.3 ]
hang 1: p1=0.749
hang 2: p1=0.800
hang 3: p1=0.300
hang 4: REVIEW (khong con nguon)


Trong Python, `NaN` nhân 0 vẫn là `NaN`, nên phép cộng có trọng số thông thường sẽ ra `NaN`. Code thay ô thiếu bằng 0 trong `safe_values`, đặt trọng số của ô đó bằng 0, rồi chia cho tổng các trọng số còn lại.

Ở hàng đầu, nhánh kém tin cậy chỉ còn trọng số 0.07, còn nhánh kia là 0.3. Kết quả 0.749 vì vậy gần với 0.9 hơn 0.1. Hàng chỉ còn một nguồn thì giữ đúng xác suất của nguồn ấy, 0.8 hoặc 0.3. Hàng mất cả hai nguồn được đánh dấu `REVIEW` để người xem lại, thay vì bịa ra một con số. Độ tin cậy ở đây được đặt bằng tay. Trong ứng dụng thật, phải định nghĩa và đo nó theo chất lượng thực tế của từng nguồn.

## Ví dụ 3: Train hai nhánh, rồi chọn cách ghép trên tập hiệu chỉnh

Hai cảm biến tổng hợp cùng đo một đại lượng ẩn, mỗi cảm biến có nhiễu riêng. Nhãn bằng 1 khi đại lượng ẩn dương. Mỗi nhánh nhận số đo của một cảm biến, cộng thêm một đặc trưng nhiễu không liên quan. Đây là bài phân loại từ hai nguồn số, chưa phải ghép ảnh với chữ.

Ta chia 300 mẫu để train, 150 mẫu để hiệu chỉnh, và 150 mẫu để đánh giá cuối. Logit của nhánh A bị cố tình nhân 3, để giả lập một model quá tự tin. Temperature và trọng số ghép được chọn trên tập hiệu chỉnh. Điểm chỉ được báo trên tập đánh giá cuối.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

rng = np.random.default_rng(74)
latent = rng.normal(size=600)
labels = (latent > 0).astype(int)
features_a = np.column_stack([latent + rng.normal(0, 0.8, 600), rng.normal(size=600)])
features_b = np.column_stack([latent + rng.normal(0, 0.8, 600), rng.normal(size=600)])
order = rng.permutation(600)
train, calibration, valid = order[:300], order[300:450], order[450:]
assert not set(train) & set(calibration) and not set(calibration) & set(valid)

models = []
for features in [features_a, features_b]:
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
    model.fit(features[train], labels[train])
    assert np.array_equal(model.classes_, [0, 1])
    models.append(model)

def sigmoid(logits):
    return 1 / (1 + np.exp(-logits))

def binary_log_loss(targets, positive_p):
    return log_loss(targets, np.column_stack([1 - positive_p, positive_p]), labels=[0, 1])

a_cal_logits = 3 * models[0].decision_function(features_a[calibration])
b_cal_p = models[1].predict_proba(features_b[calibration])[:, 1]
temperature_grid = [1.0, 2.0, 3.0, 4.0]
temperature_losses = [binary_log_loss(labels[calibration], sigmoid(a_cal_logits / t))
                      for t in temperature_grid]
temperature = temperature_grid[int(np.argmin(temperature_losses))]
a_cal_p = sigmoid(a_cal_logits / temperature)
weight_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
fusion_losses = [binary_log_loss(labels[calibration], w * a_cal_p + (1 - w) * b_cal_p)
                 for w in weight_grid]
weight_a = weight_grid[int(np.argmin(fusion_losses))]

a_valid_logits = 3 * models[0].decision_function(features_a[valid])
raw_a_p = sigmoid(a_valid_logits)
a_valid_p = sigmoid(a_valid_logits / temperature)
b_valid_p = models[1].predict_proba(features_b[valid])[:, 1]
fusion_p = weight_a * a_valid_p + (1 - weight_a) * b_valid_p
assert np.isfinite(fusion_p).all() and np.all((fusion_p >= 0) & (fusion_p <= 1))
assert np.array_equal(raw_a_p >= 0.5, a_valid_p >= 0.5)

print(f"temperature A = {temperature:.1f}; trong so A = {weight_a:.2f}")
for name, probabilities in [("A qua tu tin", raw_a_p), ("A calibrated", a_valid_p),
                            ("B", b_valid_p), ("fusion", fusion_p)]:
    print(f"{name}: accuracy={accuracy_score(labels[valid], probabilities >= 0.5):.3f}, "
          f"log-loss={binary_log_loss(labels[valid], probabilities):.3f}")

temperature A = 2.0; trong so A = 0.50
A qua tu tin: accuracy=0.793, log-loss=0.732
A calibrated: accuracy=0.793, log-loss=0.509
B: accuracy=0.787, log-loss=0.455
fusion: accuracy=0.807, log-loss=0.404


Tập hiệu chỉnh chọn temperature 2 cho nhánh A. Chia logit cho 2 làm xác suất bớt cực đoan, nhưng không đổi dấu của logit, nên các quyết định ở ngưỡng 0.5 giữ nguyên. Accuracy của A vì vậy vẫn là 0.793, còn log-loss giảm từ 0.732 xuống 0.509. Log-loss phạt nặng những dự đoán sai mà quá tự tin, nên nó thấy được cải thiện mà accuracy không thấy.

Trọng số ghép được chọn là 0.5 cho mỗi nhánh. Ghép lại cho accuracy 0.807 và log-loss 0.404, tốt hơn từng nhánh riêng, trên 150 mẫu chưa hề tham gia lựa chọn. Hai cảm biến có nhiễu độc lập nên bổ sung cho nhau. Bước hiệu chỉnh cần đủ dữ liệu riêng của nó. Và nếu dữ liệu test đến từ một phân phối khác, xác suất đã hiệu chỉnh có thể không còn tốt như vậy.